In [1]:
!pip uninstall -y torch torchvision torchaudio synthcity numpy
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2
!pip install numpy==1.26.4
!pip install synthcity==0.2.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 217.0 MB/s  0:00:03eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 227.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 246.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 234.1 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 212.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 234.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 658.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 230.9 MB/s  0:00:03eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 232.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 232.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 226.5 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 226.0 MB/s  0:00:00

In [3]:
import torch, numpy, synthcity
print("torch:", torch.__version__)
print("numpy:", numpy.__version__)
print("synthcity:", getattr(synthcity, "__version__", "unknown"))

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.2.2+cu121
numpy: 1.26.4
synthcity: unknown


In [2]:
import torch, math

if not hasattr(torch.nn, "RMSNorm"):
    class RMSNorm(torch.nn.Module):
        """Minimal RMSNorm stub for torch<2.3 compatibility."""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.eps = eps
            self.weight = torch.nn.Parameter(torch.ones(dim))

        def forward(self,   x):
            norm = x.norm(2, dim=-1, keepdim=True)
            rms = norm / math.sqrt(x.shape[-1])
            return x / (rms + self.eps) * self.weight

    torch.nn.RMSNorm = RMSNorm
    print("Added temporary torch.nn.RMSNorm shim")
else:
    print("torch.nn.RMSNorm already exists")

Added temporary torch.nn.RMSNorm shim


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
import pandas as pd
from synthcity.plugins.core.dataloader import SurvivalAnalysisDataLoader
from synthcity.plugins import Plugins

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



[KeOps] Warning : There were warnings or errors :
<stdin>:1:10: fatal error: nvrtc.h: No such file or directory
compilation terminated.

[KeOps] Warning : CUDA include path not found. Please set the CUDA_PATH or CUDA_HOME environment variable.

[KeOps] Warning : There were warnings or errors :
<stdin>:1:10: fatal error: nvrtc.h: No such file or directory
compilation terminated.

[KeOps] Warning : CUDA include path not found. Please set the CUDA_PATH or CUDA_HOME environment variable.
[KeOps] Compiling cuda jit compiler engine ... 
[KeOps] Warning : There were warnings or errors :
/opt/app-root/lib64/python3.12/site-packages/keopscore/binders/nvrtc/nvrtc_jit.cpp:19:10: fatal error: nvrtc.h: No such file or directory
   19 | #include <nvrtc.h>
      |          ^~~~~~~~~
compilation terminated.

OK
[pyKeOps] Compiling nvrtc binder for python ... 
[KeOps] Warning : There were warnings or errors :
In file included from /opt/app-root/lib/python3.12/site-packages/pykeops/common/keops_io/pyke

In [5]:
df = pd.read_csv("rotterdam_dataset.csv").copy()

In [6]:
categorical_vars = ["meno","size","grade","hormon","chemo"]

for col in categorical_vars:
    if col in df.columns:
        df[col] = df[col].astype("category")

In [7]:
if "pid" in df.columns:
    df.drop(columns=["pid"], inplace=True)

if "year" in df.columns:
    df.drop(columns=["year"], inplace=True)

In [8]:
df_os = df.copy()
if "recur" in df_os.columns:
    df_os.drop(columns=["recur"], inplace=True)

if "rtime" in df_os.columns:
    df_os.drop(columns=["rtime"], inplace=True)

In [9]:
df_os["death"] = pd.to_numeric(df_os["death"])
df_os["dtime"] = pd.to_numeric(df_os["dtime"])

In [10]:
loader_os = SurvivalAnalysisDataLoader(
    df_os,
    target_column="death",
    time_to_event_column="dtime"
)

In [11]:
plugins = Plugins(categories=["survival_analysis"])

In [20]:
model_os = plugins.get(
    "survival_gan",
    n_iter=700,
    batch_size=128,
    lr=1e-4,
    random_state=42
)

In [21]:
model_os.fit(loader_os)

[2026-03-17T17:48:41.928973+0000][50][CRITICAL] module disabled: /opt/app-root/lib64/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2026-03-17T17:48:41.929799+0000][50][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-03-17T17:48:41.930436+0000][50][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-03-17T17:48:41.931533+0000][50][CRITICAL] module plugin_great load failed
100%|██████████| 700/700 [03:43<00:00,  3.13it/s]


In [22]:
syn_df_os = model_os.generate(count=len(df_os)).dataframe()
syn_df_os.head()

,age,meno,size,grade,nodes,pgr,er,hormon,chemo,dtime,death
0,60,1,<=20,3,4,451,49,1,1,770,0
1,70,1,20-50,2,3,765,349,0,0,3472,0
2,41,0,20-50,2,0,109,103,0,0,2707,0
3,50,1,<=20,3,0,22,597,0,0,3339,0
4,30,0,20-50,3,0,13,0,0,0,3669,0


In [23]:
syn_df_os.to_csv("survivalGAN_synthetic_os.csv", index=False)

In [24]:
df_dfs = df.copy()
if "death" in df_dfs.columns:
    df_dfs.drop(columns=["death"], inplace=True)

if "dtime" in df_dfs.columns:
    df_dfs.drop(columns=["dtime"], inplace=True)

In [25]:
df_dfs["recur"] = pd.to_numeric(df_dfs["recur"])
df_dfs["rtime"] = pd.to_numeric(df_dfs["rtime"])

In [26]:
loader_dfs = SurvivalAnalysisDataLoader(
    df_dfs,
    target_column="recur",
    time_to_event_column="rtime"
)

In [33]:
model_dfs = plugins.get(
    "survival_gan",
    n_iter=620,
    batch_size=128,
    lr=1e-4,
    random_state=42
)

In [36]:
model_dfs.fit(loader_dfs)

[2026-03-17T18:18:39.313073+0000][50][CRITICAL] module disabled: /opt/app-root/lib64/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2026-03-17T18:18:39.313941+0000][50][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-03-17T18:18:39.314437+0000][50][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-03-17T18:18:39.314937+0000][50][CRITICAL] module plugin_great load failed
100%|██████████| 620/620 [03:12<00:00,  3.23it/s]


In [37]:
syn_df_dfs = model_dfs.generate(count=len(df_dfs)).dataframe()
syn_df_dfs.head()

,age,meno,size,grade,nodes,pgr,er,hormon,chemo,rtime,recur
0,51,1,<=20,2,0,614,46,0,0,2404,0
1,45,0,<=20,2,1,119,16,0,1,2577,0
2,33,0,<=20,3,0,374,68,0,0,2931,0
3,72,1,20-50,2,9,15,73,0,0,1893,0
4,43,0,<=20,3,2,12,8,0,1,884,1


In [38]:
syn_df_dfs.to_csv("survivalGAN_synthetic_dfs.csv", index=False)

In [39]:
#full synthesis
df["death"] = pd.to_numeric(df["death"])
df["dtime"] = pd.to_numeric(df["dtime"])
df["rtime"] = pd.to_numeric(df["rtime"])
df["recur"] = pd.to_numeric(df["recur"])

In [40]:
loader_full = SurvivalAnalysisDataLoader(
    df,
    target_column="death",
    time_to_event_column="dtime"
)

In [42]:
model_full = plugins.get(
    "survival_gan",
    n_iter=650,
    batch_size=128,
    lr=1e-4,
    random_state=42
)

In [43]:
model_full.fit(loader_full)

[2026-03-17T18:36:37.535337+0000][50][CRITICAL] module disabled: /opt/app-root/lib64/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2026-03-17T18:36:37.536280+0000][50][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-03-17T18:36:37.537272+0000][50][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-03-17T18:36:37.537984+0000][50][CRITICAL] module plugin_great load failed
100%|██████████| 650/650 [03:39<00:00,  2.96it/s]


In [34]:
syn_full = model_full.generate(count=len(df)).dataframe()
syn_full.head()

,age,meno,size,grade,nodes,pgr,er,hormon,chemo,rtime,recur,dtime,death
0,74,1,20-50,3,7,128,264,1,0,2116,0,2137,0
1,59,1,20-50,2,1,5,325,0,0,2499,0,2602,0
2,43,0,<=20,3,0,0,8,0,0,5623,0,5647,0
3,58,1,20-50,3,1,4,88,0,0,2619,0,2632,0
4,50,0,<=20,2,1,83,14,0,1,4908,0,4897,0


In [35]:
syn_full.to_csv("survivalGAN_synthetic_full.csv", index=False)